In [1]:
import logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
from typing import List, Any
from langchain_core.documents import Document
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

def load_all_document(data_dir:str)->List[Any]:
    """Load all documents from data directory"""
    data_path = Path(data_dir).resolve()
    logger.debug(f"data path: {data_path}")
    # print(f"[DEBUG] data path: {data_path}")
    documents = []

    ## PDFs
    pdf_files = list(data_path.glob("**/*.pdf"))
    logger.debug(f"Found {len(pdf_files)} pdf files")
    for pdf_file in pdf_files:
        try:
            logger.debug(f"Loading pdf: {pdf_file}")
            loader = PyMuPDFLoader(str(pdf_file))
            docs = loader.load()
            for doc in docs:
                doc.metadata["file_type"] = "pdf"
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata["contract_type"] = pdf_file.parent.name
                # print(doc.metadata)
            logger.debug(f"loader {len(docs)} docs from pdf: {pdf_file}")
            documents.extend(docs)
            logger.debug(f"loaded document: {pdf_file}")
        except Exception as e:
            logger.error(f"Failed to load {pdf_file}: {e}")
    logger.info(f"loaded {len(documents)}")
    return documents

c:\Users\elbou\Desktop\contractAssistant\contractAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-31 23:10:53,694 | DEBUG | PyTorch version 2.6.0+cu124 available.


In [3]:
docs = load_all_document("../data/marketing")

2026-03-31 23:11:05,924 | DEBUG | data path: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\Marketing
2026-03-31 23:11:05,925 | DEBUG | Found 1 pdf files
2026-03-31 23:11:05,926 | DEBUG | Loading pdf: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\Marketing\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf
2026-03-31 23:11:07,349 | DEBUG | loader 6 docs from pdf: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\Marketing\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf
2026-03-31 23:11:07,352 | DEBUG | loaded document: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\Marketing\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf
2026-03-31 23:11:07,353 | INFO | loaded 6


In [4]:
docs

[Document(metadata={'producer': 'Aspose.Pdf for .NET 17.6', 'creator': 'Aspose Ltd.', 'creationdate': '', 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\Marketing\\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'file_path': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\Marketing\\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-01-27T15:09:28-05:00', 'trapped': '', 'modDate': "D:20200127150928-05'00'", 'creationDate': '', 'page': 0, 'file_type': 'pdf', 'source_file': 'PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'contract_type': 'Marketing'}, page_content='Exhibit 10.6\nTRADEMARK LICENSE AGREEMENT\nThis TRADEMARK LICENSE AGREEMENT (this “Agreement”) is

### enrich metadata

In [ ]:
import re
import logging
import pandas as pd

logger = logging.getLogger(__name__)


def enrich_metadata(documents):

    logger.info(f"strat enriching metadata for {len(documents)} documents")
    data = pd.read_csv("../data/master_clauses.csv")

    for doc in documents:
        for row in data:
            print(f"{doc.metadata.get("source_file")} ||| {row["Filename"]}" )
    #         if doc.metadata.get("source_file") == row["Filename"]:
    #             #parties:
    #             parties = re.findall(r'([^;()]+)(?:\s*\(|$)', row["Parties-Answer"])
    #             parties = [party.strip() for party in parties if party.strip()] 
    #             party_1 = parties[0]
    #             party_2 = parties[1]

    #             # converting dates
    #             agre_dt_obj = pd.to_datetime(row["Agreement Date-Answer"], format='%m/%d/%y')
    #             agre__qdrant_date = agre_dt_obj.strftime('%Y-%m-%dT%H:%M:%SZ')

    #             effec_dt_obj = pd.to_datetime(row["Agreement Date-Answer"], format='%m/%d/%y')
    #             effec__qdrant_date = effec_dt_obj.strftime('%Y-%m-%dT%H:%M:%SZ')

    #             exper_dt_obj = pd.to_datetime(row["Agreement Date-Answer"], format='%m/%d/%y')
    #             exper__qdrant_date = exper_dt_obj.strftime('%Y-%m-%dT%H:%M:%SZ')

    #             doc.metadata.update({
    #                 "party_1":party_1,
    #                 "party_2":party_2,
    #                 "contract_type":row["Document Name-Answer"],
    #                 "agreement_date":agre__qdrant_date,
    #                 "effective_date":effec__qdrant_date,
    #                 "expiration_date":exper__qdrant_date,
    #                 "notice_period_to_terminate": row["Notice Period To Terminate Renewal- Answer"],
    #                 "renewl_term": row["Renewal Term-Answer"],
    #                 "governing_law": row["Governing Law-Answer"]

    #             })
    # logger.info(f"metadata enriched successfully")
    return documents


In [6]:
docs = enrich_metadata(docs)

2026-03-31 23:11:43,316 | INFO | strat enriching metadata for 6 documents


TypeError: string indices must be integers, not 'str'

In [7]:
docs

[Document(metadata={'producer': 'Aspose.Pdf for .NET 17.6', 'creator': 'Aspose Ltd.', 'creationdate': '', 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\Marketing\\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'file_path': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\Marketing\\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-01-27T15:09:28-05:00', 'trapped': '', 'modDate': "D:20200127150928-05'00'", 'creationDate': '', 'page': 0, 'file_type': 'pdf', 'source_file': 'PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'contract_type': 'Marketing', 'expiry_date': '', 'Clause_type': ['liability', 'governing_law', 'intellectual_property'], 'party_1': 'Palmer Sq

### chunking

In [ ]:
import logging
import re
from typing import List
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

logger = logging.getLogger(__name__)

SECTION_PATTERN = re.compile(r"\n\s*(\d+)\.\s+[A-Z][^\n]+")


def split_by_sections(text: str):
    """
    Split contract text by legal section headers like:
    1. Services
    2. Compensation
    """

    matches = list(SECTION_PATTERN.finditer(text))
    if not matches:
        return [text]
    sections = []
    for i, match in enumerate(matches):
        header_info = f"Section {match.group(0)}:\n"

        start = match.start()
        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)
        section_content = text[start:end].strip()
        sections.append(f"{header_info}{section_content}")
    return sections


def chunk_contract_documents(documents: List[Document]) -> List[Document]:
    """
    Chunk contracts using section-aware chunking,
    then semantic chunking for large sections.
    """
    ## add model_kwargs and encode_kwargs to force using GPU
    model_kwargs = {"device":"cuda"}
    encode_kwargs = {"normalize_embeddings": True}
    embeddings = HuggingFaceEmbeddings(
        model_name = "BAAI/bge-large-en-v1.5",
        model_kwargs = model_kwargs,
        encode_kwargs = encode_kwargs
        
    )

    semantic_chunker = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=90,
    )
    
    logger.info(f"initialise embedding model for Semantic Chunker ")
    all_chunks = []
    logger.info("start chunking ...")
    for doc in documents:
        text = doc.page_content
        # split by contract sections
        sections = split_by_sections(text)
        for section in sections:
            if len(section) < 2000:
                # small section → keep as one chunk
                chunk = Document(
                    page_content=section,
                    metadata=doc.metadata.copy()
                )
                all_chunks.append(chunk)
            else:
                # large section → semantic chunking
                semantic_chunks = semantic_chunker.create_documents(
                    [section],
                    [doc.metadata]
                )
                all_chunks.extend(semantic_chunks)

    logger.info(f"Created {len(all_chunks)} chunks")

    return all_chunks


In [9]:
chunks = chunk_contract_documents(docs)

C:\Users\elbou\AppData\Local\Temp\ipykernel_788\3589939815.py:38: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
2026-03-22 19:57:59,826 | INFO | Use pytorch device_name: cpu
2026-03-22 19:57:59,826 | INFO | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-03-22 19:57:59,839 | DEBUG | Starting new HTTPS connection (1): huggingface.co:443
2026-03-22 19:58:01,077 | DEBUG | https://huggingface.co:443 "HEAD /BAAI/bge-large-en-v1.5/resolve/main/modules.json HTTP/1.1" 307 0
2026-03-22 19:58:01,224 | DEBUG | https://huggingface.co:443 "HEAD /api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modu

[INFO] Created 16 chunks


In [10]:
chunks

[Document(metadata={'producer': 'Aspose.Pdf for .NET 17.6', 'creator': 'Aspose Ltd.', 'creationdate': '', 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\Marketing\\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'file_path': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\Marketing\\PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'total_pages': 6, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-01-27T15:09:28-05:00', 'trapped': '', 'modDate': "D:20200127150928-05'00'", 'creationDate': '', 'page': 0, 'file_type': 'pdf', 'source_file': 'PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'contract_type': 'Marketing', 'Clause_type': 'liability', 'parties': []}, page_content='Exhibit 10.6\nTRADEMARK LICENSE AGREEMENT\nThis TRADEM

In [ ]:
# # delete collection

# from qdrant_client import QdrantClient

# client = QdrantClient(host="localhost", port=6333)
# client.delete_collection(collection_name="contracts")  # deletes old collection

c:\Users\elbou\Desktop\contractAssistant\contractAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### store in the qdrant db

In [ ]:
## creating data
import logging
import hashlib
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models

logger = logging.getLogger(__name__)


## embeed then store
class QdrantStore:

    def __init__(
        self,
        dense_model_name: str = "BAAI/bge-large-en-v1.5",
        sparse_model_name: str = "Qdrant/bm25",
    ):
        self.dense_model_name = dense_model_name
        self.sparse_model_name = sparse_model_name
        ## add model_kwargs={'device': 'cuda'} for GPU Enabling
        self.dense_model = SentenceTransformer(model_name = self.dense_model_name,device="cuda")
        ## add providers=["CUDAExecutionProvider"] for GPU Enabling
        self.saprse_model = SparseTextEmbedding(model_name=self.sparse_model_name, providers=["CUDAExecutionProvider"])
        self.client = QdrantClient(host="localhost", port=6333)
        self.collection_name = "contracts_v1"
        self.creat_collection()

    def creat_collection(self):
        if not self.client.collection_exists(collection_name="contracts"):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config={
                    "dense": models.VectorParams(
                        size=1024, distance=models.Distance.COSINE
                    )
                },
                sparse_vectors_config={
                    "sparse": models.SparseVectorParams(
                        index=models.SparseIndexParams(on_disk=False)
                    )
                },
            )
    ## generate IDs
    def generate_doc_id(self, source: str, content: str):
        unique_string = f"{source}:{content}"
        return hashlib.sha256(unique_string.encode()).hexdigest()[:32]

    def embedde_chunks_and_store(self, chunks):

        logger.info(f"generating embedding {len(chunks)} chunks ...")

        texts = [chunk.page_content for chunk in chunks]

        dense_embeddings = self.dense_model.encode(
            texts, normalize_embeddings=True, show_progress_bar=True
        )
        sparse_embeddings = list(self.saprse_model.embed(texts))

        # Focuses on Semantic Direction: By removing the effect of vector magnitude, normalization ensures that similarity is based on semantic meaning rather than frequency or length.
        logger.info(f"embeddings shape: {dense_embeddings.shape}")

        points = []
        batch_size = 64

        for chunk, dense_embedding, sparse_embedding in zip(
            chunks, dense_embeddings, sparse_embeddings
        ):
            ## generating ID
            source = chunk.metadata.get("source_file","uknown")
            doc_id = self.generate_doc_id(source,chunk.page_content)

            ## getting general context
            contract_type =chunk.metadata.get("contract_type","Legal Document")
            parties = f"{chunk.metadata.get('party_1','')} and {chunk.metadata.get('party_2','')}"
            points.append(
                models.PointStruct(
                    # id=uuid.uuid4(),
                    id=doc_id,
                    vector={
                        "dense": dense_embedding.tolist(),
                        "sparse": models.SparseVector(
                            indices=sparse_embedding.indices,
                            values=sparse_embedding.values,
                        ),
                    },
                    payload={
                        "text": f"Document: {contract_type} between {parties}, \n Context: {chunk.page_content}",
                        ## filename
                        "source": chunk.metadata.get("source_file", ""),
                        "file_type": chunk.metadata.get("file_type", ""),
                        "page": chunk.metadata.get("page", ""),
                        "contract_type": chunk.metadata.get("contract_type", ""),
                        # "clause_type": chunk.metadata.get("Clause_type", ""),
                        "agreement_date":chunk.metadata.get("agreement_date", ""),
                        "effective_date": chunk.metadata.get("effective_date", ""),
                        "expiration_date": chunk.metadata.get("expiration_date", ""),
                        "party_1": chunk.metadata.get("party_1", ""),
                        "party_2": chunk.metadata.get("party_2", ""),
                        "notice_period_to_terminate": chunk.metadata.get("notice_period_to_terminate", ""),
                        "renewl_term": chunk.metadata.get("renewl_term", ""),
                        "governing_law": chunk.metadata.get("governing_law", ""),
                    },
                )
            )
            if len(points) > batch_size:
                self.client.upsert(collection_name=self.collection_name, points=points)
                logger.info(f"upsertted {len(points)}..")
                points = []
        if points:
            self.client.upsert(collection_name=self.collection_name, points=points)
            logger.info(f"upsertted final {len(points)} points.")

        logger.info("chunks stored in Qdrant.")


In [13]:
store = QdrantStore()
store.embedde_chunks_and_store(chunks)

2026-03-22 19:59:47,850 | INFO | Use pytorch device_name: cpu
2026-03-22 19:59:47,850 | INFO | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-03-22 19:59:48,818 | DEBUG | https://huggingface.co:443 "HEAD /BAAI/bge-large-en-v1.5/resolve/main/modules.json HTTP/1.1" 307 0
2026-03-22 19:59:49,918 | DEBUG | https://huggingface.co:443 "HEAD /api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json HTTP/1.1" 200 0
2026-03-22 19:59:50,853 | DEBUG | https://huggingface.co:443 "HEAD /BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json HTTP/1.1" 307 0
2026-03-22 19:59:51,565 | DEBUG | https://huggingface.co:443 "HEAD /api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json HTTP/1.1" 200 0
2026-03-22 19:59:51,826 | DEBUG | https://huggingface.co:443 "HEAD /BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json HTTP/1.1" 307 0
2026-03-22 

### rewrite query

In [ ]:

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
import os 

load_dotenv()

class QueryRewriting:
    def __init__(self, model_name:str="allam-2-7b"): #
        self.model_name = model_name

        groq_api_key = os.getenv("GROQ_API_KEY")
        if not groq_api_key:
            raise ValueError("GROQ_API_KEY not found in environment variables")
        self.llm = ChatGroq(groq_api_key = groq_api_key, model_name = self.model_name, temperature=0)

        self.prompt = PromptTemplate(
            input_variables=["original_query"],
            template="""You are an expert legal assistant helping retrieve information \
                    from enterprise contracts.

                    Reformulate the user query below to be more precise and retrieval-friendly.
                    Focus on: legal terminology, clause names, date references.
                    Return ONLY the rewritten query, nothing else.

                    Original query: {original_query}
                    Rewritten query:"""
        )

        self.chain = self.prompt | self.llm
        print(f"[INFO] Groq LLM initialized {self.model_name}")

    def rewrite_query(self,original_query:str):
        if not original_query.strip():
            raise ValueError("original query cannot be empty")
        response = self.chain.invoke({"original_query": original_query})
        return response.content.strip()


In [66]:
original_query = "this agreement is made by who ?"
rewriter = QueryRewriting()
rewriten = rewriter.rewrite_query(original_query)

2026-03-22 23:52:49,201 | DEBUG | Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-a8afb55d-7e63-4959-b87e-e5df94cab4e7', 'json_data': {'messages': [{'role': 'user', 'content': 'You are an expert legal assistant helping retrieve information                     from enterprise contracts.\n\n                    Reformulate the user query below to be more precise and retrieval-friendly.\n                    Focus on: legal terminology, clause names, date references.\n                    Return ONLY the rewritten query, nothing else.\n\n                    Original query: this agreement is made by who ?\n                    Rewritten query:'}], 'model': 'allam-2-7b', 'n': 1, 'reasoning_effort': None, 'reasoning_format': None, 'service_tier': 'on_demand', 'stop': None, 'stream': False, 'temperature': 1e-08}}
2026-03-22 23:52:49,201 | DEBUG | Sending HTTP Request: POST https://api.groq.com/openai/v1/chat/compl

[INFO] Groq LLM initialized allam-2-7b


2026-03-22 23:52:58,449 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000023C69ED7BD0>
2026-03-22 23:52:58,452 | DEBUG | start_tls.started ssl_context=<ssl.SSLContext object at 0x0000023C69D82450> server_hostname='api.groq.com' timeout=None
2026-03-22 23:53:02,868 | DEBUG | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000023C69E86950>
2026-03-22 23:53:02,868 | DEBUG | send_request_headers.started request=<Request [b'POST']>
2026-03-22 23:53:02,872 | DEBUG | send_request_headers.complete
2026-03-22 23:53:02,872 | DEBUG | send_request_body.started request=<Request [b'POST']>
2026-03-22 23:53:02,872 | DEBUG | send_request_body.complete
2026-03-22 23:53:02,872 | DEBUG | receive_response_headers.started request=<Request [b'POST']>
2026-03-22 23:53:19,801 | DEBUG | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Sun, 22 Mar 2026 22:53:02 GMT'), (b'Content-Type', b'applicat

### filters

In [ ]:

from ingestion.vectorStore import QdrantStore
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_groq import ChatGroq
import os
import logging
from dotenv import load_dotenv

load_dotenv()
logger = logging.getLogger(__name__)

def get_filter_from_query(query:str):
    metadata_info = [
        AttributeInfo(
            name="source_file",
            description="The name of the document such as 'CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'. Use this to filter by name of document",
            type="string"
        ),
        AttributeInfo(
            name="party_1",
            description="The name of the first legal entity or company mentioned in the contract. Use this to filter by the primary party or signer.",
            type="string"
        ),
        AttributeInfo(
            name="party_2",
            description="The name of the second legal entity or counterparty in the contract. Use this to filter for the second participant in the agreement.",
            type="string"
        ),
        AttributeInfo(
            name="contract_type",
            description="The category of the agreement, such as 'Franchise Agreement', 'NDA', 'Service Agreement', or 'Lease'. Use this when the user specifies a document type.",
            type="string"
        ),
        AttributeInfo(
            name="agreement_date",
            description="The date the contract was signed or formally created. Use for questions about when a contract was made.",
            type="string"
        ),
        AttributeInfo(
            name="effective_date",
            description="The official start date of the contract's obligations. Use for questions about when an agreement becomes active.",
            type="string"
        ),
        AttributeInfo(
            name="expiration_date",
            description="The date the contract ends or becomes invalid. Use for questions about renewals, terminations, or end dates.",
            type="string"
        ),
        AttributeInfo(
            name="governing_law",
            description="The legal jurisdiction or state laws that apply to the contract (e.g., 'California', 'New York', 'Morocco'). Use this when the user mentions a specific location or law.",
            type="string"
        ),
    ]


    document_content_description = "Detailed clauses and legal text from corporate contracts"

    groq_api_key = os.getenv("GROQ_API_KEY")
    llm = ChatGroq(model_name = "", groq_api_key = groq_api_key,temperature=0)
    vectorestore = QdrantStore()

    retriever = SelfQueryRetriever.from_llm(
        llm=llm,
        vectorstore = vectorestore,## to-do later
        document_contents=document_content_description,
        metadata_field_info=metadata_info
    )

    result = retriever.invoke(query)

    filters = models.Filter(
        must=[
            models.FieldCondition(key="source_file", match=models.matchValue(result.source_file)),
            models.Filter(
                should=[
                    models.FieldCondition(key="party_1", match=models.MatchValue(result.party_1)),
                    models.FieldCondition(key="party_2", match=models.MatchValue(result.party_2))
                ]
            ),
            models.FieldCondition(key="contract_type", match=models.matchValue(result.contract_type)),
            models.FieldCondition(key="agreement_date", match=models.matchValue(result.agreement_date)),
            models.FieldCondition(key="effective_date", match=models.matchValue(result.effective_date)),
            models.FieldCondition(key="expiration_date", match=models.matchValue(result.expiration_date)),
            models.FieldCondition(key="governing_law", match=models.matchValue(result.governing_law)),
        ]
    )

    return filters





In [68]:
filters = get_filter_from_query("When does this agreement terminate?")
filters

2026-03-22 23:54:08,537 | INFO | Extracting filters from: 'When does this agreement terminate?'
2026-03-22 23:54:08,538 | INFO |   No contract type detected — searching all types


### hybrid search

In [69]:

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, FusionQuery, Fusion, Filter
from fastembed import SparseTextEmbedding
from typing import Optional
from dataclasses import dataclass


# result schema
@dataclass
class SearchResult:
    chunk_id:str
    # text:str
    score:float
    metadata:dict 


class HybridSearch:
    def __init__(self, dense_model_name:str= "BAAI/bge-large-en-v1.5", sparse_model_name:str="Qdrant/bm25"):
        self.dense_model_name = dense_model_name
        self.dense_model = SentenceTransformer(self.dense_model_name)
        self.sparse_model_name = sparse_model_name
        self.sparse_model = SparseTextEmbedding(self.sparse_model_name)

        self.client = QdrantClient(host="localhost", port=6333)
        self.collection_name = "contracts"
#   !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    def hybrid_search_with_rrf(self, query:str, filters:Optional[Filter]=None):
        """Perform hybrid search using Reciprocal Rank Fusion"""
        # rrf is a method for merging and ranking results from multiple search techniques
        query_dense_embedding = self.dense_model.encode(query,normalize_embeddings=True)
        query_sparse_embeding = list(self.sparse_model.embed(query))[0]
      
        query_sparse_vector = models.SparseVector(
            indices=query_sparse_embeding.indices,
            values=query_sparse_embeding.values
        )
      ## hybrifd search
        results = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[
                Prefetch(
                    query=query_dense_embedding,
                    using="dense",
                    limit=10,
                    filter=filters
                ),
                Prefetch(
                    query=query_sparse_vector,
                    using="sparse",
                    limit=10,
                    filter=filters
                )
            ],
            query= FusionQuery(fusion=Fusion.RRF)
        )
        return self.hybrid_search_points_to_results(results.points)
    
    def hybrid_search_points_to_results(self, points):

        return [
            SearchResult(
                chunk_id=str(point.id),
                # text= point.payload.get("text",""),
                score = point.score,
                metadata=point.payload
            )
            for point in points
        ]



In [70]:
hybrid_search = HybridSearch()
results = hybrid_search.hybrid_search_with_rrf(query=rewriten,filters=filters)

2026-03-22 23:54:09,202 | INFO | Use pytorch device_name: cpu
2026-03-22 23:54:09,204 | INFO | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-03-22 23:54:09,207 | DEBUG | Resetting dropped connection: huggingface.co
2026-03-22 23:54:15,092 | DEBUG | https://huggingface.co:443 "HEAD /BAAI/bge-large-en-v1.5/resolve/main/modules.json HTTP/1.1" 307 0
2026-03-22 23:54:18,453 | DEBUG | https://huggingface.co:443 "HEAD /api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json HTTP/1.1" 200 0
2026-03-22 23:54:21,717 | DEBUG | https://huggingface.co:443 "HEAD /BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json HTTP/1.1" 307 0
2026-03-22 23:54:30,445 | DEBUG | https://huggingface.co:443 "HEAD /api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json HTTP/1.1" 200 0
2026-03-22 23:54:39,464 | DEBUG | https://huggingface.co:443 "HEAD /BAAI/bge-large-en-

In [71]:
results

[SearchResult(chunk_id='73fb16ff-9b3b-4e80-9f60-374a6ec34acc', score=1.0, metadata={'text': 'Mutual Representations. Each party hereby represents and warrants to the other party as follows:\n(a) Due Authorization. Such party is a limited liability company duly formed or a corporation duly incorporated, as applicable, and\nis in good standing as of the Effective Date, and the execution, delivery and performance of this Agreement by such party have been duly\nauthorized by all necessary action on the part of such party. (b) Due Execution. This Agreement has been duly executed and delivered by such party and, upon due authorization, execution\nand delivery of this Agreement by the other party, constitutes a legal, valid and binding obligation of such party, enforceable against such\nparty in accordance with its terms. (c) No Conflict. Such party’s execution, delivery and performance of this Agreement do not: (i) violate, conflict with or result in\nthe breach of any provision of the opera

### rerank

In [72]:
import logging
from dataclasses import dataclass
import os 
from dotenv import load_dotenv
import cohere
from typing import Optional

load_dotenv()
logger = logging.getLogger(__name__)

@dataclass
class RerankedResult:
    chunk_id: str
    # text: str
    rerank_score: float
    metadata: dict
    original_rank: int
 

class Reranker:
    # rerank-v3.5 / rerank-v4.0
    #rerank-english-v3.0
    def __init__(self, model_name:str = "rerank-v3.5"):
        self.model_name = model_name
        cohere_api_key = os.getenv("COHERE_API_KEY")
        if not cohere_api_key:
            raise ValueError("cohere api key not found in the .env file")
        self.client = cohere.Client(cohere_api_key)
        logger.info(f"Reranker initialized - model: {model_name}")

    def rerank(self, query: str, results: list, top_n:int=5)->list:
        """
        Rerank hybrid search results using Cohere API. 
        Returns:
            list of RerankedResult sorted by rerank_score descending
        """
        if not results:
            return []
        documents = [r.metadata.get("text","") for r in results] # r.page_content for r in results

        response = self.client.rerank(
            model = self.model_name,
            query = query,
            documents = documents,
            top_n = top_n,
            return_documents = False
        )

        reranked = []
        for hit in response.results:
            original = results[hit.index]
            reranked.append(RerankedResult(
                chunk_id=original.chunk_id,
                rerank_score=hit.relevance_score,
                metadata=original.metadata,
                original_rank=hit.index + 1,
            ))
        logger.info(f"Cohere reranked {len(results)} → top {len(reranked)} results")

        return reranked




In [73]:
reranker = Reranker()
results = reranker.rerank(rewriten,results)

2026-03-23 00:00:35,830 | INFO | Reranker initialized - model: rerank-v3.5
2026-03-23 00:00:35,830 | DEBUG | connect_tcp.started host='api.cohere.com' port=443 local_address=None timeout=300 socket_options=None
2026-03-23 00:00:41,974 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000023C69DE7390>
2026-03-23 00:00:41,975 | DEBUG | start_tls.started ssl_context=<ssl.SSLContext object at 0x0000023C69E9FE30> server_hostname='api.cohere.com' timeout=300
2026-03-23 00:00:42,809 | DEBUG | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000023C6A04A810>
2026-03-23 00:00:42,809 | DEBUG | send_request_headers.started request=<Request [b'POST']>
2026-03-23 00:00:42,809 | DEBUG | send_request_headers.complete
2026-03-23 00:00:42,809 | DEBUG | send_request_body.started request=<Request [b'POST']>
2026-03-23 00:00:42,809 | DEBUG | send_request_body.complete
2026-03-23 00:00:42,809 | DEBUG | receive_response_headers.start

In [74]:
results

[RerankedResult(chunk_id='a2fa0208-f9d0-4194-b333-a6eac0afb477', rerank_score=0.2395187, metadata={'text': 'IN WITNESS WHEREOF, each party has caused this Agreement to be executed on the date first set forth above by its duly authorized\nofficer.\nLICENSOR:\nPALMER SQUARE CAPITAL\nMANAGEMENT LLC\nBy:\nName: Jeffrey D. Fox\nTitle:\nManaging Director\nLICENSEE:\nPALMER SQUARE CAPITAL BDC INC.\nBy:\nName: Scott A. Betz\nTitle:\nChief Compliance Officer\nACKNOWLEDGED AND AGREED TO\nPALMER SQUARE BDC ADVISOR LLC\nBy:\nName: Jeffrey D. Fox\nTitle:\nChief Financial Officer\n[Signature Page to Trademark License Agreement]\nSource: PALMER SQUARE CAPITAL BDC INC., 10-12G/A, 1/16/2020', 'source': 'PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf', 'file_type': 'pdf', 'page': 5, 'contract_type': 'Marketing', 'clause_type': 'intellectual_property', 'effective_date': '', 'expiry_date': '', 'parties': []}, original_rank=4),
 RerankedResult(chunk_id='

## Generation

## question answering

In [75]:

from typing import List, Dict


SYSTEM_PROMPT = """You are a precise legal contract analyst assistant for an enterprise consulting firm.
Your job is to answer questions about contracts strictly based on the provided contract excerpts.

RULES YOU MUST FOLLOW:
1. ONLY answer using information found in the provided contract excerpts below.
2. If the answer is not in the excerpts, say exactly: "This information was not found in the provided contract documents."
3. NEVER invent dates, values, clause numbers, or party names.
4. ALWAYS cite the source file and section for every claim you make.
5. Be concise and direct — answer in plain English, not legal jargon.
6. If multiple contracts are provided, specify WHICH contract you are answering from.

"""


def build_user_prompt(query:str, chunks:List[Dict])-> str:

    texts = [chunk.get("text","") for chunk in chunks]
    context = "\n\n".join(texts)
    return f"""{context}
    ---
    QUESTION: {query} 
    Answer based strictly on the excerpts above."""


## llm client

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import json

import os
from dotenv import load_dotenv


load_dotenv()


class LLMClient:

    def __init__(self, model_name: str = "openai/gpt-oss-safeguard-20b"):
        """Groq LLM client for contract question answering."""
        self.model_name = model_name
        groq_api_key = os.getenv("GROQ_API_KEY")
        if not groq_api_key:
            raise ValueError("can't find the GROQ_API_KEY in .env")
        self.llm = ChatGroq(
            model=self.model_name, groq_api_key=groq_api_key, temperature=0
        )

        print(f"[INFO] LlmClient initialized - model {self.model_name}")

    def generate_response(self, query: str, chunks):
        USER_PROMPT = build_user_prompt(query, chunks)

        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=USER_PROMPT),
        ]
        try:
            answer = self.llm.invoke(messages)
        except Exception as e:
            raise RuntimeError(f"Groq API call failed: {e}") from e

        sources = [
            {
                "filename": chunk.get("source", ""),
                "file_type": chunk.get("file_type", ""),
                "page": chunk.get("page", ""),
                "contract_type": chunk.get("contract_type", ""),
                "clause_type": chunk.get("Clause_type", ""),
                "effective_date": chunk.get("effective_date", ""),
                "expiry_date": chunk.get("expiry_date", ""),
                "parties": chunk.get("parties", ""),
                "preview": chunk["text"][0:200] + "...",
            }
            for chunk in chunks
        ]

        results = {
            "answer": answer.content,
            "sources": sources,
            "confidence": chunks[0].get("rerank_score", 0) if chunks else None,
        }
        return self.format_result(results)

    @staticmethod
    def format_result(results):

        sources_map = {}

        for source in sources_map["sources"]:
            filename = source.get("filename", "unknown")

            if filename not in sources_map:
                # First time seeing this file → add it
                sources_map[filename] = {
                    "filename": filename,
                    "contract_type": source.get("contract_type", ""),
                    "effective_date": source.get("effective_date", ""),
                    "expiry_date": source.get("expiry_date", ""),
                    "parties": source.get("parties", []),
                    "clause_types": (
                        [source.get("clause_type")] if source.get("clause_type") else []
                    ),
                    "pages": [source["page"]] if source.get("page") else [],
                    "preview": source.get("preview", ""),
                }
            else:
                # Already seen this file → just add new page and clause_type
                existing = sources_map[filename]

                if source.get("page") and source["page"] not in existing["pages"]:
                    existing["pages"].append(source["page"])

                if (
                    source.get("clause_type")
                    and source["clause_type"] not in existing["clause_types"]
                ):
                    existing["clause_types"].append(source["clause_type"])

        return {
            "answer": results["answer"],
            "sources": list(sources_map.values()),
            "confidence": results["confidence"],
        }

    # ── Chunk formatter ───────────────────────────────────────────────────────
    @staticmethod
    def reranked_to_chunks(reranked_results: list) -> list[dict]:
        """
        Convert RerankedResult objects from reranker.py into
        the simple dict format expected by build_user_prompt().

        Args:
            reranked_results : list of RerankedResult from Reranker.rerank()

        Returns:
            list of dicts with keys: text, file_name, section_title, chunk_id
        """
        return [
            {
                **r.metadata,
                "chunk_id": r.chunk_id,
                "text": r.metadata.get("text", ""),
                "file_name": r.metadata.get("source", "unknown"),
                "original_score": r.original_rank,
                "rerank_score": r.rerank_score,
            }
            for r in reranked_results
        ]

    # if __name__ == "__main__":


#     llm_client = LLMClient()
#     reranker = Reranker()
#     query_rewriter = QueryRewriting()
#     hybrid_search = HybridSearch()
#     original_query = "what is bla bla bla ?"

#     qdrant_filters = get_filter_from_query(original_query)
#     rewrited_query = query_rewriter.rewrite_query(original_query)
#     results = hybrid_search.hybrid_search_with_rrf(rewrited_query,fliters=qdrant_filters)
#     reranked_results = reranker.rerank(rewrited_query, results)
#     chunks = llm_client.reranked_to_chunks(reranked_results)

#     answer = llm_client.generate_response(rewrited_query,chunks)

#     print(answer)

SyntaxError: incomplete input (3517133214.py, line 99)

In [77]:
llm = LLMClient()

[INFO] LlmClient initialized - model openai/gpt-oss-safeguard-20b


In [78]:
chunks = llm.reranked_to_chunks(results)

In [79]:
chunks

[{'text': 'IN WITNESS WHEREOF, each party has caused this Agreement to be executed on the date first set forth above by its duly authorized\nofficer.\nLICENSOR:\nPALMER SQUARE CAPITAL\nMANAGEMENT LLC\nBy:\nName: Jeffrey D. Fox\nTitle:\nManaging Director\nLICENSEE:\nPALMER SQUARE CAPITAL BDC INC.\nBy:\nName: Scott A. Betz\nTitle:\nChief Compliance Officer\nACKNOWLEDGED AND AGREED TO\nPALMER SQUARE BDC ADVISOR LLC\nBy:\nName: Jeffrey D. Fox\nTitle:\nChief Financial Officer\n[Signature Page to Trademark License Agreement]\nSource: PALMER SQUARE CAPITAL BDC INC., 10-12G/A, 1/16/2020',
  'source': 'PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf',
  'file_type': 'pdf',
  'page': 5,
  'contract_type': 'Marketing',
  'clause_type': 'intellectual_property',
  'effective_date': '',
  'expiry_date': '',
  'parties': [],
  'chunk_id': 'a2fa0208-f9d0-4194-b333-a6eac0afb477',
  'file_name': 'PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949

In [80]:
answer = llm.generate_response(original_query,chunks)

2026-03-23 00:00:47,940 | DEBUG | Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-1a206826-81fb-4cf8-84d2-fd89f6c5718f', 'json_data': {'messages': [{'role': 'system', 'content': 'You are a precise legal contract analyst assistant for an enterprise consulting firm.\nYour job is to answer questions about contracts strictly based on the provided contract excerpts.\n\nRULES YOU MUST FOLLOW:\n1. ONLY answer using information found in the provided contract excerpts below.\n2. If the answer is not in the excerpts, say exactly: "This information was not found in the provided contract documents."\n3. NEVER invent dates, values, clause numbers, or party names.\n4. ALWAYS cite the source file and section for every claim you make.\n5. Be concise and direct — answer in plain English, not legal jargon.\n6. If multiple contracts are provided, specify WHICH contract you are answering from.\n\n'}, {'role': 'user', 'cont

In [81]:
answer

{'answer': 'The agreement is between **Palmer Square Capital Management LLC** (the Licensor) and **Palmer Square Capital BDC Inc.** (the Licensee).  \nThis is shown by the signature block: “LICENSOR: PALMER SQUARE CAPITAL MANAGEMENT LLC … By: Jeffrey D. Fox” and “LICENSEE: PALMER SQUARE CAPITAL BDC INC. … By: Scott A. Betz”【source: provided excerpt, signature page】',
 'sources': [{'filename': 'PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf',
   'file_type': 'pdf',
   'page': 5,
   'contract_type': 'Marketing',
   'clause_type': '',
   'effective_date': '',
   'expiry_date': '',
   'parties': [],
   'preview': 'IN WITNESS WHEREOF, each party has caused this Agreement to be executed on the date first set forth above by its duly authorized\nofficer.\nLICENSOR:\nPALMER SQUARE CAPITAL\nMANAGEMENT LLC\nBy:\nName: Jeffr...'},
  {'filename': 'PalmerSquareCapitalBdcInc_20200116_10-12GA_EX-10.6_11949289_EX-10.6_Trademark License Agreement.pdf'

In [82]:
ContractAnswer(answer='The agreement terminates if the Investment Advisor or an affiliate stops serving as an investment adviser to the Licensee; if Licensor or Licensee receives notice of a Third Party Claim related to the Licensed Mark; if either party gives sixty‑day written notice to the other; or if Licensee attempts to assign or sublicense the Agreement without Licensor’s prior written consent.', confidence='high', citations=[Citation(file_name='PALMER SQUARE CAPITAL BDC INC., 10-12G/A, 1/16/2020', relevant_quote='This Agreement shall expire if the Investment Advisor or one of its affiliates ceases to serve as investment adviser to the Licensee.'), Citation(file_name='PALMER SQUARE CAPITAL BDC INC., 10-12G/A, 1/16/2020', relevant_quote='This Agreement shall be terminable by Licensor at any time and in its sole discretion in the event that Licensor or Licensee receives notice of any Third Party Claim… or by Licensor or Licensee upon sixty (60) days’ written notice to the other party… or by Licensee at any time in the event Licensee assigns or attempts to assign or sublicense…')], intent='termination', expiry_date=None, contract_value=None, notice_period_days=60)

NameError: name 'ContractAnswer' is not defined